# L32 · NumPy 信号基础 — `np.arange` / `linspace`，生成 16kHz 一秒时间轴

**今日目标**：写出 `time_axis(duration, sample_rate)` 函数，理解四个变量之间的关系。

**核心关系**：
- `N = round(duration × sample_rate)` — 采样点总数
- `t = np.arange(N) / sample_rate` — 每个采样点对应的时刻（秒）

Aurora Audio Core 读取任何音频信号的第一步都是建立这条时间轴；后续的 STFT 和 mel 滤波器组以同样的点序号换算时间偏移。

看到 ✏️ 的格子就是你要填代码的地方，按 `Shift+Enter` 逐格运行。

## 本课剧情：给声音铺轨道

声音在计算机里是一列数字，每个数字代表一个采样点的振幅。`np.arange(N)` 先生成点的整数序号 `0, 1, 2, ..., N-1`，除以 `sample_rate` 就把序号换算成秒——这是后续所有信号处理的基础坐标系。本节任务是写出 `time_axis(duration, sample_rate)` 并用 assert 验证它。

### 写 `time_axis` 前明确三件事

- **输入**：`duration`（总时长，秒）、`sample_rate`（每秒采样点数，Hz）
- **关键步骤**：先算 `N = round(duration * sample_rate)`，再算 `np.arange(N) / sample_rate`
- **返回**：长度为 `N` 的一维 float 数组，每个元素是对应采样点的时刻（秒）

## 实验入口：把声音拆成可观察的数组

用短时长（0.1 s）和低采样率（16 Hz）运行第一个例子，直接打印 `N` 和 `t`，观察点数如何随 `duration × sample_rate` 变化，以及 `t[-1]` 总是比 `duration` 短一个采样间隔。

## 1. 导入 numpy

`import numpy as np` = 把数值计算库 numpy 拿进来，并起个简称 `np`。以后 `np.xxx` 就是在用它。

你可以把 numpy 想成"会整批处理数字的工具箱"。普通 Python 适合一个一个处理值，numpy 适合一次处理一整串值。我们今天要生成的是"序列"，所以它正好派上用场。运行下面这格：

**为什么用 `np.arange(N) / sample_rate` 而不是 `np.linspace(0, duration, N)`？** `arange` 生成严格整数序号，除以采样率后每步长恰好是 `1/sample_rate`，不存在浮点累积误差。`linspace` 在端点之间均匀插值，N 很大时中间值与"第 k 点的精确时刻 `k/sample_rate`"可能有微小偏差——这个差异在 FFT 频率对齐时会被放大。另外，`t[-1]` 等于 `(N-1)/sample_rate`，比 `duration` 少一个步长：N 个点的编号是 `0` 到 `N-1`，最后一点的时刻不是 `duration` 本身，这是正确的。


In [2]:
import numpy as np
print('numpy 已就绪:', np.__version__)

numpy 已就绪: 2.5.0


## 动手观察：序列怎样一步步变成信号

修改 `sample_rate` 或 `duration`，观察 `n`（整数序号数组）和 `t`（时间数组）的长度如何同步变化。重点确认：`t[-1]` 始终等于 `(N-1) / sample_rate`，而不是 `duration` 本身。

In [ ]:
import numpy as np

sample_rate = 8
duration = 1.0
freq = 2.0
N = round(duration * sample_rate)
n = np.arange(N)
t = n / sample_rate
angle = 2 * np.pi * freq * t
wave = np.sin(angle)

print('N =', N)
print('采样点编号 n =', n)
print('时间轴 t =', np.round(t, 3))
print('角度 angle =', np.round(angle, 3))
print('sin(angle) =', np.round(wave, 3))


## 代码实验：遍历不同时长和采样率

验证 `N = duration × sample_rate` 在不同参数组合下严格成立，同时确认 `t[-1]` 始终比总时长短一个采样间隔 `1 / sample_rate`。

In [ ]:
import numpy as np

for duration in [0.25, 0.5, 1.0]:
    for sample_rate in [4, 8]:
        N = round(duration * sample_rate)
        t = np.arange(N) / sample_rate
        print(f'duration={duration}, sample_rate={sample_rate} -> N={N}, t={np.round(t, 3)}')


## 2. 先认识 `np.arange`

`np.arange(n)` 生成一串整数 `0, 1, 2, ..., n-1`。这一步非常关键，因为它先帮我们搭出“第几个采样点”的骨架。

可以把它看成：先拿到一排座位号，再决定每个座位对应的具体时刻。对数组做 `/ 数字` 会**每个元素都除**，所以我们能把整数序号直接变成时间。先玩一下：


In [3]:
print(np.arange(5))          # [0 1 2 3 4]
print(np.arange(5) / 10)     # [0.  0.1 0.2 0.3 0.4]  每个都除以 10

[0 1 2 3 4]
[0.  0.1 0.2 0.3 0.4]


## 3. ✏️ 你的任务：实现 `time_axis`

时间轴 = 每个采样点对应的时刻（秒）。如果 1 秒里有 16000 个点，那么第 0 个点是 `0` 秒，第 1 个点是 `1/16000` 秒，第 2 个点是 `2/16000` 秒，以此类推。

这里最容易糊涂的地方有两个：
1. "一共多少点" 不是 `duration`，而是 `duration * sample_rate`
2. 时间轴不是先写一堆小数，而是先写"点的编号"，再除以采样率

**推理路线**：
1. 算出总采样点数：`N = round(duration * sample_rate)`——用 `round` 是因为浮点乘法可能产生非整数，比如 `0.1 * 8000` 在 Python 里是 `799.9999...`
2. 生成整数编号序列：`n = np.arange(N)`，结果是 `[0, 1, 2, ..., N-1]`
3. 把编号换算成秒：`t = n / sample_rate`，每个元素恰好是"第几个点 ÷ 每秒点数"

**参考输入输出**：`duration=0.5, sample_rate=8` → N=4，n=[0,1,2,3]，t=[0, 0.125, 0.25, 0.375]

<details><summary>点击查看参考实现</summary>

```python
def time_axis(duration, sample_rate):
    N = round(duration * sample_rate)
    n = np.arange(N)
    return n / sample_rate
```

</details>

In [9]:
def time_axis(duration, sample_rate):
    # ✏️ TODO: 返回时间轴 [0, 1/sr, 2/sr, ...]，长度 = duration*sample_rate
    return np.arange(duration * sample_rate) / sample_rate

## 4. 运行检查（自动判卷）

这一步不是“机器替你思考”，而是帮你快速确认三件事：长度对不对、前几个值对不对、最后一个值落在哪。

实现对了，下面会打印 ✅；错了会报错告诉你哪不对。建议你先看报错信息，再回去对照上面的变量关系，而不是只盯着答案本身。


In [10]:
sr = 16000
t = time_axis(1.0, sr)
print('采样点个数  :', t.shape[0], ' (期望', sr, ')')
print('前 5 个时刻 :', t[:5])
print('最后 1 个时刻:', round(float(t[-1]), 6), '秒')

assert t.shape[0] == sr, '采样点个数应等于采样率'
assert abs(t[1] - t[0] - 1/sr) < 1e-12, '相邻时刻间隔应为 1/sr'
print('\n✅ 通过：采样率与时间轴关系检查完成。')


采样点个数  : 16000  (期望 16000 )
前 5 个时刻 : [0.000e+00 6.250e-05 1.250e-04 1.875e-04 2.500e-04]
最后 1 个时刻: 0.999938 秒

✅ 通过：你理解了采样率如何决定时间轴。


## 5. 逐行回顾（看懂了就过）

- `np.arange(duration * sample_rate)` → 先得到一串“点编号”，比如 `0,1,2,...,15999`
- `/ sample_rate` → 再把编号换算成秒，所以每个元素都像“这一点在时间轴上的位置”
- `t.shape[0]` = 数组长度；`t[:5]` = 前 5 个；`t[-1]` = 最后一个，这些都是看序列是否符合预期的基本方法
- `assert 条件, '信息'` = 自动判卷：真则放行，假则报错
- `abs(...) < 1e-12`：电脑算小数有微小误差，所以判断“足够接近”而不是“严格相等”

你可以把这整节课记成一句话：**先造整数序列，再把整数序列映射成时间序列。** 这就是很多音频、信号、图像代码的基本思路。

**🎉 完成后**：保存 notebook（`Ctrl+S`），回终端 `git add notebooks/week01/day1_numpy.ipynb && git commit -m 'learn: Day 1'`


## 🎨 图示：信号 = 一串采样点

In [11]:
from aurora.audio import sine
import aviz; aviz.style()
aviz.waveform(sine(5.0, duration=1.0, sample_rate=64), stem=True,
              title='5 Hz 正弦 @ 64 采样点');

ModuleNotFoundError: No module named 'aurora'

In [ ]:
duration_options = [0.125, 0.25, 0.5]
sample_rates = [8, 16]

for duration in duration_options:
    for sr in sample_rates:
        N = round(duration * sr)
        t = np.arange(N) / sr
        verdict = '空序列' if len(t) == 0 else f'最后一点={t[-1]:.4f}s'
        print(f'duration={duration:>5}, sr={sr:>2} -> N={N:>2}, {verdict}')


## 参数实验：采样率翻倍的效果

把 `sample_rate` 从 `8` 改到 `16`（采样率翻倍），对比 `duration=0.5` 下的结果：

| sample_rate | N | t[-1] |
|---|---|---|
| 8 | 4 | 0.375 s |
| 16 | 8 | 0.4375 s |

N 和 t 数组长度都翻倍，但两个 `t[-1]` 都接近 `duration=0.5` 而没有超出——更高采样率用更多点覆盖同样时长，而不是覆盖更长时段。每次只改一个参数，先预测 N 和 `t[-1]` 的值，再运行验证。

In [ ]:
duration_options = [0.125, 0.25, 0.5]
sample_rates = [8, 16]

for duration in duration_options:
    for sr in sample_rates:
        N = round(duration * sr)
        t = np.arange(N) / sr
        last = '无最后点' if len(t) == 0 else f'{t[-1]:.4f}s'
        print(f'duration={duration:>5}, sr={sr:>2} -> N={N:>2}, last={last}')


## 本课收束

现在能用 `time_axis(duration, sample_rate)` 生成任意时长的时间轴数组，并通过 assert 验证长度和端点值。这个函数对应 Aurora Audio Core 中所有信号读取路径的第一步——在进入任何频域分析之前都需要先建立这个坐标系。Day 2 会把今天的 `t` 代入 `np.sin(2 * np.pi * freq * t)`，生成实际的正弦波音频信号。

In [ ]:
# 小检查：从短序列开始，确认每一步输出
import numpy as np

sample_rate = 8
duration = 1.0
freq = 1.0
N = round(duration * sample_rate)
n = np.arange(N)
t = n / sample_rate
angle = 2 * np.pi * freq * t
x = np.sin(angle)

print('1) N 应该是多少？', N)
print('2) n 是采样点编号：', n)
print('3) t 是每个点的秒数：', np.round(t, 3))
print('4) angle 是每个点在圆上的角度：', np.round(angle, 3))
print('5) x 是最终波形：', np.round(x, 3))
